In [ ]:
import os
from dlisio import dlis
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from db import db

# Loading wells

Each well is a folder, named by its directory name. Inside each well folder, there are multiple DLIS files. Each DLIS file is loaded and processed to extract its frames and channels, which are then stored in the database.

In [ ]:
wells = {}
with os.scandir("wells") as entries:
    for entry in entries:
        path = entry.path
        wells[entry.name] = [f for f in os.listdir(path)]

In [ ]:
LOG_TYPES = {
  "MUDLOG": 1,
  "LWD": 2
}

In [ ]:
def well_deserializer(well_name):
    well_id = db.create_well(well_name)
    
    for file in wells[well_name]:
        if not file.endswith(".DLIS"):
            continue
        
        with dlis.load(f"wells/{well_name}/{file}") as (f, *tail):
            log_type_id = LOG_TYPES["LWD"] # Default log type (there are more LWD files than MUD files)  
            
            if "MUD" in file.upper():
                log_type_id = LOG_TYPES["MUDLOG"]

            for frame in f.frames:
                file_id = db.find_or_create("logical_file", {
                    "well_id": well_id,
                    "name": file,
                    "log_type_id": log_type_id
                })

                curves = frame.curves()
                index_array = curves[frame.index]
                
                for c in frame.channels:
                    if c.name == frame.index:
                        continue
                    
                    channel_id = db.find("channel", c.name, c.units)
                    
                    if channel_id is None:
                        channel_id = db.create_channel(
                            file_id, c.name, c.units
                        )

                    for i, reading_val in enumerate(curves[c.name]):
                        index_val = index_array[i]

                        if reading_val == -999.25: # DLIS missing value indicator
                            reading_val = None

                        db.create_reading(well_id, file_id, channel_id, index_val, reading_val)

In [ ]:
for well_name in wells.keys():
    well_deserializer(well_name)